In [5]:
%pip install openpyxl 


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd   
import time
t0 = time.time()
df = pd.read_excel('Реестр (1).xlsx')
print(time.time()-t0)

C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


KeyboardInterrupt: 

In [24]:
import zipfile    
with zipfile.ZipFile('Реестр (1).xlsx') as z:
    for info in z.infolist():
        print(info.filename, info.file_size)

_rels/.rels 603
[Content_Types].xml 1095
docProps/app.xml 183
docProps/core.xml 445
xl/sharedStrings.xml 137
xl/styles.xml 2644
xl/workbook.xml 352
xl/_rels/workbook.xml.rels 581
xl/worksheets/sheet1.xml 1455301080


In [22]:
with zipfile.ZipFile('Реестр (1).xlsx') as z:        
    xml_bytes = z.read("xl/worksheets/sheet1.xml")
print(xml_bytes[:300].decode("utf-8"))

<?xml version="1.0" encoding="UTF-8"?>
<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main"><dimension ref="A1"/><sheetViews><sheetView workbookViewId="0" tabSelected="true"/></sheetViews><sheetFormatPr defaultRowHeight="15.0"/><cols><col min="1" max="1" width="7.140625" cust


In [13]:
import zipfile  
import re
import shutil
import time

SOURCE_FILE = "Реестр (1).xlsx"
FIXED_FILE = "Реестр_fixed.xlsx"

t0 = time.time()

with zipfile.ZipFile(SOURCE_FILE, "r") as zin, \
     zipfile.ZipFile(FIXED_FILE, "w", zipfile.ZIP_DEFLATED) as zout:

    for item in zin.infolist():
        if item.filename == "xl/worksheets/sheet1.xml":
            with zin.open(item) as fin, zout.open(item.filename, "w") as fout:
                first_chunk = fin.read(2000)
                first_chunk = re.sub(
                    rb'<dimension ref="[^"]*"/>',
                    b'<dimension ref="A1:W2000000"/>',
                    first_chunk,
                    count=1,
                )
                fout.write(first_chunk)
                shutil.copyfileobj(fin, fout, length=1024 * 1024)
        else:
            zout.writestr(item, zin.read(item.filename))

print(f"Готово за {time.time() - t0:.1f} сек: {FIXED_FILE}")

Готово за 48.9 сек: Реестр_fixed.xlsx


In [15]:
import openpyxl  
import time

t0 = time.time()

wb = openpyxl.load_workbook("Реестр_fixed.xlsx", read_only=True)
ws = wb["Лист1"]

count = 0
for i, row in enumerate(ws.iter_rows(values_only=True)):
    if i < 6:              
        print(i, row)
    count += 1

print("Всего строк:", count)
print("Время:", time.time() - t0, "сек")

0 ('Единый реестр субъектов малого и среднего предпринимательства по состоянию на 15.04.2024', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
1 (None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None)
2 ('№ п/п', 'Наименование / ФИО', 'Тип субъекта', 'Категория', 'ОГРН', 'ИНН', 'Основной вид деятельности', 'Регион', 'Район', 'Город', 'Населенный пункт', 'Вновь созданный', 'Дата включения в реестр', 'Дата исключения из реестра', 'Телефон', 'E-mail', 'WWW', 'Наличие лицензий', 'Наличие заключенных договоров, контрактов', 'Производство инновационной, высокотехнологичной продукции', 'Участие в программах партнерства', 'Является социальным предприятием', 'Среднесписочная численность работников за предшествующий календарный год')
3 (1.0, '" МЕГАПОЛИС " ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ', 'Юридическое лицо', 'Не является 

In [16]:
import re  
import csv 

okved_pattern = re.compile(r"^41\.20?\b")

allowed_categories = {"Малое предприятие", "Среднее предприятие"}

wb = openpyxl.load_workbook("Реестр_fixed.xlsx", read_only=True)
ws = wb["Лист1"]

header = None
matched_rows = []
total_scanned = 0

for i, row in enumerate(ws.iter_rows(values_only=True)):
    if i == 2:
        header = row
        continue
    if i < 3:
        continue

    total_scanned += 1

    subject_type = row[2]  
    category = row[3]      
    okved = row[6]         

    if subject_type != "Юридическое лицо":
        continue
    if category not in allowed_categories:
        continue
    if not okved or not okved_pattern.match(okved.strip()):
        continue

    matched_rows.append(row)

wb.close()

print("Просмотрено строк:", total_scanned)
print("Подошло под условия:", len(matched_rows))

Просмотрено строк: 603520
Подошло под условия: 10573


In [17]:
with open('filtered_construction.csv','w',newline = '', encoding = 'utf-8') as f:  
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(matched_rows)
print('Сохранено: ', len(matched_rows), 'строк')

Сохранено:  10573 строк


In [20]:
import pandas as pd  

df = pd.read_csv("filtered_construction.csv")
print(df.shape)
df

(10573, 23)


,№ п/п,Наименование / ФИО,Тип субъекта,Категория,ОГРН,ИНН,Основной вид деятельности,Регион,Район,Город,...,Дата исключения из реестра,Телефон,E-mail,WWW,Наличие лицензий,"Наличие заключенных договоров, контрактов","Производство инновационной, высокотехнологичной продукции",Участие в программах партнерства,Является социальным предприятием,Среднесписочная численность работников за предшествующий календарный год
0,20.0,"""КОРПОРАЦИЯ ВИТ"" (ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕ...",Юридическое лицо,Малое предприятие,1025004907080,5038038838,41.20 Строительство жилых и нежилых зданий,50 - Московская область,Пушкино г,NaN,...,NaN,NaN,NaN,NaN,Нет,Нет,Нет,Нет,Нет,36.0
1,25.0,"""ЛАВИНА"" ОБЩЕСТВО С ОГРАНИЧЕННОЙ ОТВЕТСТВЕННОСТЬЮ",Юридическое лицо,Малое предприятие,1035010952624,5027064258,41.2 Строительство жилых и нежилых зданий,50 - Московская область,NaN,г Люберцы,...,NaN,NaN,NaN,NaN,Нет,Нет,Нет,Нет,Нет,40.0
2,52.0,"""ХОЗРАСЧЕТНАЯ СТРОИТЕЛЬНО-ТЕХНОЛОГИЧЕСКАЯ ФИРМ...",Юридическое лицо,Среднее предприятие,1025007270551,5027006369,41.20 Строительство жилых и нежилых зданий,50 - Московская область,NaN,г Дзержинский,...,NaN,NaN,NaN,NaN,Да,Нет,Нет,Нет,Нет,201.0
3,4651.0,"АКЦИОНЕРНОЕ ОБЩЕСТВО ""2МЕН ГРУПП ДЕВЕЛОПМЕНТ""",Юридическое лицо,Малое предприятие,1067746424899,7701651356,41.2 Строительство жилых и нежилых зданий,72 - Тюменская область,NaN,г Тюмень,...,NaN,NaN,NaN,NaN,Да,Нет,Нет,Нет,Нет,19.0
4,4653.0,"АКЦИОНЕРНОЕ ОБЩЕСТВО ""777""",Юридическое лицо,Малое предприятие,1021400692048,1414006922,41.20 Строительство жилых и нежилых зданий,77 - г.Москва,NaN,NaN,...,NaN,NaN,NaN,NaN,Да,Нет,Нет,Нет,Нет,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10568,560903.0,СОВМЕСТНОЕ ПРЕДПРИЯТИЕ ОБЩЕСТВО С ОГРАНИЧЕННОЙ...,Юридическое лицо,Среднее предприятие,1026500785673,6504043928,41.20 Строительство жилых и нежилых зданий,65 - Сахалинская область,Корсаковский р-н,г Корсаков,...,NaN,NaN,NaN,NaN,Да,Нет,Нет,Нет,Нет,145.0
10569,565172.0,СТРОИТЕЛЬНО-ИНВЕСТИЦИОННАЯ КОМПАНИЯ ОБЩЕСТВО С...,Юридическое лицо,Малое предприятие,1032307179409,2312105041,41.2 Строительство жилых и нежилых зданий,23 - Краснодарский край,NaN,г Краснодар,...,NaN,NaN,NaN,NaN,Нет,Нет,Нет,Нет,Нет,16.0
10570,576233.0,"УПРАВЛЕНИЕ МЕХАНИЗИРОВАННЫХ РАБОТ "" КАСКАД "" (...",Юридическое лицо,Малое предприятие,1022300509637,2301032458,41.2 Строительство жилых и нежилых зданий,23 - Краснодарский край,Анапский р-н,NaN,...,NaN,NaN,NaN,NaN,Нет,Нет,Нет,Нет,Нет,24.0
10571,580133.0,"ФИРМА ""ТЕПЛОСТРОЙПРОЕКТ-С"" ОБЩЕСТВО С ОГРАНИЧЕ...",Юридическое лицо,Малое предприятие,1032000400233,2002001476,41.20 Строительство жилых и нежилых зданий,20 - Чеченская Республика,Ачхой-Мартановский р-н,NaN,...,NaN,NaN,NaN,NaN,Да,Нет,Нет,Нет,Нет,67.0
